# 00 · RecGym Data Audit

**Project:** Sensor-Efficient Wearable Gym Exercise Recognition: An Axis-Level and Sensor-Placement Analysis Under Subject-Independent Evaluation

**Purpose of this notebook:** inspect the *real* RecGym data before any modelling decision is made. It trains **no model** and reports **no performance numbers**. Everything it prints or saves is descriptive and comes from the file it loads.

**How to run (Colab or Kaggle):** `Runtime → Run all`. At the end it zips `outputs/` and (on Colab) triggers a download.

**What it saves (inside `outputs/`)**

| Path | Content |
|---|---|
| `data_audit_report.md` | Human-readable summary of every check, with PASS / WARN / FAIL flags |
| `audit_summary.json` | Machine-readable version of the same facts |
| `label_map.json`, `position_map.json`, `subject_list.json` | Actual labels / positions / subjects found in the file (no assumed labels) |
| `axis_configs.json` | The 15 axis configurations of the primary experiment (single source of truth for later notebooks) |
| `loso_folds.json` | Fixed LOSO fold definition (train subjects / test subject) |
| `compute_plan.json` | Number of model fits implied by the design |
| `tables/*.csv` | Coverage, run-length, window-feasibility, signal-statistics tables |
| `figures/*.png` | Diagnostic plots |
| `cli_test_sample.csv.gz` | Small contiguous sample so your CLI agent can unit-test later code locally |
| `environment.json`, `audit_log.txt` | Versions, seed, dataset SHA-256 |

**Rules this notebook respects:** no random row-level splitting, no model fitting, no feature or hyper-parameter selection. Window-size candidates are only checked for *feasibility* (enough windows per class and subject); the final window size must be fixed from prior reasoning, not from test performance.

## 1 · Configuration

In [ ]:
import os, sys, re, json, glob, time, hashlib, platform, random, shutil, zipfile, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=FutureWarning)

# ============================ USER CONFIG ====================================
SEED = 42
DATA_PATH = os.environ.get("RECGYM_CSV", "data/RecGym.csv")   # path to RecGym.csv; None or "" = auto-find / auto-download
RESULTS_DIR = Path(os.environ.get("RESULTS_DIR", "results/raw/data_audit"))   # where this notebook writes its outputs
UCI_ID = 1128                                    # RecGym on the UCI repository
SAMPLING_RATE_HZ = 20                            # from dataset documentation; NOT verifiable from the file (no timestamp column)
WINDOW_SECONDS_CANDIDATES = [2, 3, 4, 5]         # feasibility check only
OVERLAP_CANDIDATES = [0.0, 0.5]
MIN_WINDOWS_PER_SUBJECT_CLASS = 5                # threshold used only to raise a WARN flag

# Column overrides. Leave as None to auto-detect; set a string if detection fails.
COL_SUBJECT = None
COL_LABEL = None
COL_POSITION = None
COL_CHANNELS = None        # e.g. {"Ax":"A_x","Ay":"A_y","Az":"A_z","Gx":"G_x","Gy":"G_y","Gz":"G_z","C1":"C_1"}

SAVE_CANONICAL_PARQUET = True    # compact copy for later notebooks (saved in cache/, NOT zipped by default)
INCLUDE_CACHE_IN_ZIP = False
CLI_SAMPLE_RUNS_PER_GROUP = 2    # runs per (subject, position, label) in the small CLI test sample
CLI_SAMPLE_ROWS_PER_RUN = 200
# ==============================================================================

IN_KAGGLE = os.path.exists("/kaggle/working")
IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")
# On Colab/Kaggle keep using the session working directory; otherwise use the
# RESULTS_DIR configured above (relative to the repository root by default).
BASE = Path("/kaggle/working") if IN_KAGGLE else (Path("/content") if IN_COLAB else Path(".").resolve())
OUT = Path(RESULTS_DIR) if (not IN_KAGGLE and not IN_COLAB) else (BASE / "outputs")
CACHE = BASE / "cache"
for d in [OUT, OUT / "tables", OUT / "figures", CACHE]:
    d.mkdir(parents=True, exist_ok=True)

random.seed(SEED); np.random.seed(SEED)

LOG_LINES, FLAGS = [], []          # FLAGS: (status, check, detail)
SUMMARY = {}

def log(*a):
    s = " ".join(str(x) for x in a)
    print(s)
    LOG_LINES.append(s)

def flag(status, check, detail=""):
    FLAGS.append((status, check, detail))
    log(f"[{status}] {check}" + (f" — {detail}" if detail else ""))

def save_table(df, name, index=True):
    p = OUT / "tables" / f"{name}.csv"
    df.to_csv(p, index=index)
    return p

def save_fig(fig, name):
    p = OUT / "figures" / f"{name}.png"
    fig.savefig(p, dpi=130, bbox_inches="tight")
    plt.close(fig)
    return p

def save_json(obj, name):
    with open(OUT / name, "w") as f:
        json.dump(obj, f, indent=2, default=str)

log("Base dir:", BASE.resolve(), "| Colab:", IN_COLAB, "| Kaggle:", IN_KAGGLE)

## 2 · Locate and load the data
Order tried: your `DATA_PATH` → files already present → direct UCI download → `ucimlrepo` → manual upload (Colab).

In [ ]:
def find_local():
    cands = []
    if DATA_PATH:
        cands.append(DATA_PATH)
    cands += glob.glob("/kaggle/input/**/*ecGym*.csv", recursive=True)
    cands += glob.glob(str(BASE / "*ecGym*.csv")) + glob.glob("./*ecGym*.csv")
    cands += glob.glob(str(CACHE / "*ecGym*.csv"))
    for c in cands:
        if c and os.path.exists(c):
            return c
    return None

def download_uci_zip():
    import urllib.request
    url = ("https://archive.ics.uci.edu/static/public/1128/"
           "recgym+gym+workouts+recognition+dataset+with+imu+and+capacitive+sensor.zip")
    zpath = CACHE / "recgym_uci.zip"
    urllib.request.urlretrieve(url, zpath)
    with zipfile.ZipFile(zpath) as z:
        z.extractall(CACHE)
    hits = glob.glob(str(CACHE / "**" / "*ecGym*.csv"), recursive=True)
    if not hits:
        raise FileNotFoundError("zip downloaded but no RecGym csv inside")
    return hits[0]

def fetch_with_ucimlrepo():
    os.system(f"{sys.executable} -m pip install -q ucimlrepo")
    from ucimlrepo import fetch_ucirepo
    ds = fetch_ucirepo(id=UCI_ID)
    df = ds.data.original if getattr(ds.data, "original", None) is not None else pd.concat([ds.data.features, ds.data.targets], axis=1)
    p = CACHE / "RecGym_ucimlrepo.csv"
    df.to_csv(p, index=False)
    return str(p)

data_file, how = find_local(), "local file"
if data_file is None:
    for name, fn in [("direct UCI zip", download_uci_zip), ("ucimlrepo", fetch_with_ucimlrepo)]:
        try:
            data_file, how = fn(), name
            break
        except Exception as e:
            log(f"Auto-download via {name} failed: {type(e).__name__}: {e}")
if data_file is None and IN_COLAB:
    try:
        from google.colab import files
        log("Please upload RecGym.csv now ...")
        up = files.upload()
        data_file = str(BASE / list(up.keys())[0]); how = "manual upload"
        with open(data_file, "wb") as f: f.write(list(up.values())[0])
    except Exception as e:
        log("Manual upload failed:", e)
assert data_file is not None, ("RecGym.csv not found. Set DATA_PATH in the config cell "
                               "(or add the UCI dataset as a Kaggle input / upload the file).")

def sha256_of(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk)
            if not b: break
            h.update(b)
    return h.hexdigest()

DATA_SHA256 = sha256_of(data_file)
log("Data file:", data_file, f"({how})")
log("Size (MB):", round(os.path.getsize(data_file) / 1e6, 2), "| SHA-256:", DATA_SHA256)

t0 = time.time()
raw = pd.read_csv(data_file)
raw.columns = [str(c).strip() for c in raw.columns]
log("Loaded", raw.shape, "in", round(time.time() - t0, 1), "s")
log("Columns:", list(raw.columns))
log("dtypes:\n" + raw.dtypes.to_string())
display(raw.head(8))
SUMMARY["data_file"] = data_file
SUMMARY["data_sha256"] = DATA_SHA256
SUMMARY["raw_shape"] = list(raw.shape)
SUMMARY["raw_columns"] = list(raw.columns)

## 3 · Detect column roles
Nothing is assumed silently: detected roles are printed, and you can override them in the config cell.

In [ ]:
def norm(c): return re.sub(r"[^a-z0-9]", "", str(c).lower())
ncol = {norm(c): c for c in raw.columns}

def pick(cands, override):
    if override is not None:
        assert override in raw.columns, f"Override '{override}' not in columns {list(raw.columns)}"
        return override
    for k in cands:
        if k in ncol: return ncol[k]
    return None

subj_col = pick(["object", "subject", "subjectid", "volunteer", "user", "participant", "id"], COL_SUBJECT)
label_col = pick(["workout", "label", "activity", "exercise", "class"], COL_LABEL)
pos_col = pick(["position", "placement", "location", "bodyposition"], COL_POSITION)

wanted = {"Ax": "ax", "Ay": "ay", "Az": "az", "Gx": "gx", "Gy": "gy", "Gz": "gz", "C1": "c1"}
if COL_CHANNELS is not None:
    chan_cols = dict(COL_CHANNELS)
else:
    chan_cols = {k: ncol.get(v) for k, v in wanted.items()}

missing_roles = [n for n, v in [("subject", subj_col), ("label", label_col), ("position", pos_col)] if v is None]
missing_chan = [k for k, v in chan_cols.items() if v is None]
assert not missing_roles and not missing_chan, (
    f"Could not detect: roles={missing_roles}, channels={missing_chan}. "
    f"Columns are {list(raw.columns)}. Set COL_* overrides in the config cell.")

IMU = ["Ax", "Ay", "Az", "Gx", "Gy", "Gz"]
ALL_CH = IMU + ["C1"]
log("Subject column :", subj_col)
log("Label column   :", label_col)
log("Position column:", pos_col)
log("Channel columns:", chan_cols)
SUMMARY["columns_used"] = {"subject": subj_col, "label": label_col, "position": pos_col, "channels": chan_cols}

# Extra, unexpected columns?
used = {subj_col, label_col, pos_col, *chan_cols.values()}
extra = [c for c in raw.columns if c not in used]
log("Unused extra columns:", extra if extra else "none")
if extra:
    flag("WARN", "Unexpected extra columns present", str(extra))

## 4 · Build the canonical frame (file order preserved)
Rows are **never sorted or shuffled** — file order is the only time information available, and Section 7 tests whether it behaves like time.

In [ ]:
df = pd.DataFrame({
    "subject": raw[subj_col].astype(str).str.strip(),
    "position": raw[pos_col].astype(str).str.strip(),
    "label": raw[label_col].astype(str).str.strip(),
})
for k in ALL_CH:
    df[k] = pd.to_numeric(raw[chan_cols[k]], errors="coerce").astype("float32")
df["row_idx"] = np.arange(len(df), dtype="int64")
n_rows = len(df)
del raw

# --- subject-column sanity (UCI metadata lists it as "Binary"; verify what it really is)
subs = sorted(df["subject"].unique(), key=lambda s: (len(s), s))
log("Unique subject values:", len(subs), subs[:20])
SUMMARY["n_subjects"] = len(subs)
if len(subs) == 10:
    flag("PASS", "10 subjects found", "matches dataset documentation")
else:
    flag("WARN", f"{len(subs)} subjects found (documentation says 10)", "check the subject column carefully before using LOSO")

# --- positions / labels (raw values, repr() exposes whitespace / case variants)
pos_vals = df["position"].value_counts()
lab_vals = df["label"].value_counts()
log("Raw position values:", [repr(x) for x in pos_vals.index])
log("Raw label values   :", [repr(x) for x in lab_vals.index])
for name, s in [("position", pos_vals), ("label", lab_vals)]:
    folded = {}
    for v in s.index: folded.setdefault(re.sub(r"[\s_\-]+", "", v.lower()), []).append(v)
    clashes = {k: v for k, v in folded.items() if len(v) > 1}
    if clashes:
        flag("WARN", f"{name}: spelling/case variants of the same name", str(clashes))
    else:
        flag("PASS", f"{name}: no spelling/case variants")

labels = sorted(lab_vals.index.tolist())
positions = sorted(pos_vals.index.tolist())
label_map = {l: i for i, l in enumerate(labels)}
position_map = {p: i for i, p in enumerate(positions)}
null_like = [l for l in labels if "null" in l.lower()]
log("Labels (sorted):", labels)
log("Positions      :", positions)
log("Null-like labels:", null_like if null_like else "none found")
SUMMARY.update({"labels": labels, "positions": positions, "null_like_labels": null_like, "n_rows": n_rows})
save_json(label_map, "label_map.json"); save_json(position_map, "position_map.json"); save_json(subs, "subject_list.json")

if len(labels) == 12: flag("PASS", "12 labels found", "matches documentation (11 workouts + null)")
else: flag("WARN", f"{len(labels)} labels found (documentation says 12)", "update the label list used downstream")
if len(positions) == 3: flag("PASS", "3 positions found", str(positions))
else: flag("WARN", f"{len(positions)} positions found (documentation says 3)", str(positions))

df["label_id"] = df["label"].map(label_map).astype("int16")

## 5 · Integrity: missing values, infinities, duplicates, constant channels

In [ ]:
na = df[ALL_CH].isna().sum()
inf = np.isinf(df[ALL_CH].to_numpy(dtype="float64")).sum(axis=0)
integ = pd.DataFrame({"n_nan": na.values, "n_inf": inf,
                      "std": df[ALL_CH].std().values,
                      "min": df[ALL_CH].min().values, "max": df[ALL_CH].max().values}, index=ALL_CH)
display(integ)
save_table(integ, "integrity_by_channel")
if int(na.sum()) + int(inf.sum()) == 0: flag("PASS", "No NaN / inf in any channel")
else: flag("FAIL", "NaN/inf present", f"NaN={int(na.sum())}, inf={int(inf.sum())} — decide handling BEFORE modelling (never impute using test-subject data)")

n_dup = int(df.duplicated(subset=["subject", "position", "label"] + ALL_CH).sum())
log("Exact duplicate rows (same subject/position/label/all channels):", n_dup, f"({100*n_dup/n_rows:.3f}%)")
SUMMARY["n_duplicate_rows"] = n_dup
flag("PASS" if n_dup / n_rows < 0.001 else "WARN", "Duplicate rows", f"{n_dup} ({100*n_dup/n_rows:.3f}%) — not removed; row order is preserved")

const = [k for k in ALL_CH if integ.loc[k, "std"] == 0]
flag("PASS" if not const else "FAIL", "Constant channels", str(const) if const else "none")

## 6 · Coverage: subjects × labels × positions
LOSO needs every class to exist in the training subjects, and macro-F1 is only fully defined for classes that exist in the held-out subject.

In [ ]:
cnt = df.groupby(["position", "subject", "label"]).size().rename("n_rows").reset_index()
cnt["minutes_at_nominal_rate"] = cnt["n_rows"] / SAMPLING_RATE_HZ / 60
save_table(cnt, "coverage_position_subject_label", index=False)

# rows per position per subject — do the three positions record the same amount?
ps = df.groupby(["subject", "position"]).size().unstack("position").reindex(subs)
display(ps)
save_table(ps, "rows_per_subject_per_position")
equal_rows = bool((ps.nunique(axis=1) == 1).all())
if equal_rows: flag("PASS", "Each subject has equal row counts across positions")
else: flag("WARN", "Row counts differ across positions for some subjects", "positions may be unsynchronised; do not assume row i is the same instant across positions")

# overall label counts and imbalance
lab_tab = df.groupby(["label", "position"]).size().unstack("position").fillna(0).astype(int)
lab_tab["total"] = lab_tab.sum(axis=1)
lab_tab["share_%"] = (100 * lab_tab["total"] / lab_tab["total"].sum()).round(2)
display(lab_tab)
save_table(lab_tab, "label_counts_by_position")
imb_all = lab_tab["total"].max() / lab_tab["total"].min()
non_null = lab_tab.drop(index=null_like, errors="ignore")
imb_nn = non_null["total"].max() / non_null["total"].min()
log(f"Imbalance ratio (max/min class rows): all labels={imb_all:.2f}, excluding null-like={imb_nn:.2f}")
SUMMARY["imbalance_ratio_all"] = float(imb_all); SUMMARY["imbalance_ratio_excl_null"] = float(imb_nn)

# presence matrices
missing_combo = []
for p in positions:
    pres = cnt[cnt.position == p].pivot(index="subject", columns="label", values="n_rows").reindex(index=subs, columns=labels)
    absent = pres.isna().stack()
    for (s, l), v in absent.items():
        if v: missing_combo.append((p, s, l))
if missing_combo:
    flag("WARN", f"{len(missing_combo)} (position, subject, label) combinations are absent", str(missing_combo[:10]) + (" ..." if len(missing_combo) > 10 else ""))
else:
    flag("PASS", "Every subject has every label at every position")
SUMMARY["absent_combinations"] = missing_combo

# LOSO feasibility: class absent from all training subjects?
loso_issue = []
for p in positions:
    sub_cnt = cnt[cnt.position == p]
    for held in subs:
        train_labels = set(sub_cnt[sub_cnt.subject != held].label)
        test_labels = set(sub_cnt[sub_cnt.subject == held].label)
        unseen = sorted(test_labels - train_labels)
        if unseen: loso_issue.append((p, held, unseen))
if loso_issue: flag("FAIL", "Some held-out classes never appear in the training subjects", str(loso_issue[:5]))
else: flag("PASS", "In every LOSO fold, every test class appears in the training subjects")

# heatmap of minutes (nominal rate) per subject x label, one panel per position
fig, axes = plt.subplots(1, len(positions), figsize=(5.2 * len(positions), 0.55 * len(subs) + 3), squeeze=False)
for ax, p in zip(axes[0], positions):
    m = cnt[cnt.position == p].pivot(index="subject", columns="label", values="minutes_at_nominal_rate").reindex(index=subs, columns=labels)
    im = ax.imshow(m.values, aspect="auto"); ax.set_title(f"{p}: minutes @ {SAMPLING_RATE_HZ} Hz")
    ax.set_xticks(range(len(labels))); ax.set_xticklabels(labels, rotation=75, ha="right", fontsize=7)
    ax.set_yticks(range(len(subs))); ax.set_yticklabels(subs, fontsize=7)
    fig.colorbar(im, ax=ax, fraction=0.04)
save_fig(fig, "coverage_heatmap_subject_label")

fig, ax = plt.subplots(figsize=(9, 4))
lab_tab.drop(columns=["total", "share_%"]).plot(kind="bar", ax=ax)
ax.set_ylabel("rows"); ax.set_title("Rows per label and position")
save_fig(fig, "label_counts_by_position")

## 7 · Is file order a valid time order?
The file has no timestamp column. Windowing is only valid if consecutive rows inside a `(subject, position, label)` block are consecutive samples. We test this through **run structure**: a *run* is a maximal block of consecutive rows with the same subject, position and label. If the file had been shuffled, runs would be tiny.

In [ ]:
s_code = pd.factorize(df["subject"])[0]; p_code = pd.factorize(df["position"])[0]; l_code = df["label_id"].to_numpy()
chg = np.ones(n_rows, dtype=bool)
chg[1:] = (s_code[1:] != s_code[:-1]) | (p_code[1:] != p_code[:-1]) | (l_code[1:] != l_code[:-1])
starts = np.flatnonzero(chg)
lens = np.diff(np.r_[starts, n_rows])
runs = pd.DataFrame({"start_row": starts, "length": lens,
                     "subject": df["subject"].to_numpy()[starts],
                     "position": df["position"].to_numpy()[starts],
                     "label": df["label"].to_numpy()[starts]})
runs["seconds_nominal"] = runs["length"] / SAMPLING_RATE_HZ
log("Number of runs:", len(runs), "| median run length (rows):", int(runs.length.median()),
    "| min:", int(runs.length.min()), "| max:", int(runs.length.max()))
SUMMARY["n_runs"] = int(len(runs)); SUMMARY["median_run_len_rows"] = float(runs.length.median())

by_lab = runs.groupby("label")["length"].agg(n_runs="count", median="median", min="min", max="max", total="sum")
by_lab["median_seconds"] = by_lab["median"] / SAMPLING_RATE_HZ
display(by_lab)
save_table(by_lab, "run_length_by_label")
save_table(runs, "runs_all", index=False)

# Is each (subject, position) stored as ONE contiguous block, or interleaved?
sp = s_code.astype(np.int64) * 100 + p_code
sp_chg = np.r_[True, sp[1:] != sp[:-1]]
n_blocks, n_groups = int(sp_chg.sum()), len(np.unique(sp))
log(f"(subject, position) blocks in file order: {n_blocks} for {n_groups} groups")
SUMMARY["subject_position_blocks"] = n_blocks; SUMMARY["subject_position_groups"] = n_groups
if n_blocks == n_groups:
    flag("PASS", "Each (subject, position) is one contiguous block in the file")
else:
    flag("WARN", "(subject, position) data is split across several file blocks",
         f"{n_blocks} blocks vs {n_groups} groups — possibly several sessions concatenated; no session column exists to separate them")

med = runs.length.median()
if med >= 5 * SAMPLING_RATE_HZ:
    flag("PASS", "File order looks temporal", f"median run = {med:.0f} rows (~{med/SAMPLING_RATE_HZ:.0f}s at nominal rate)")
elif med >= 20:
    flag("WARN", "Runs are short", f"median run = {med:.0f} rows; windows will be limited by run length")
else:
    flag("FAIL", "File order does not look temporal", f"median run = {med:.0f} rows — windowing may be invalid; investigate before modelling")

fig, ax = plt.subplots(figsize=(9, 4))
order = by_lab.sort_values("median").index
data = [runs.loc[runs.label == l, "seconds_nominal"].to_numpy() for l in order]
ax.boxplot(data, showfliers=False); ax.set_xticks(range(1, len(order) + 1)); ax.set_xticklabels(order); ax.set_yscale("log")
ax.set_ylabel(f"run length (s @ {SAMPLING_RATE_HZ} Hz nominal)"); plt.setp(ax.get_xticklabels(), rotation=75, ha="right", fontsize=8)
ax.set_title("Run-length distribution per label")
save_fig(fig, "run_length_by_label")

## 8 · Window feasibility (counts only — no model involved)
For each candidate window length and overlap, count label-pure windows that stay **inside a run**. This tells you whether a window size is *feasible* for every subject and class. It does **not** choose the window size: pick it from prior reasoning (literature, exercise repetition duration), record the reason, and never from test performance.

In [ ]:
def n_windows(length, W, S):
    return np.where(length >= W, (length - W) // S + 1, 0)

rows = []
for sec in WINDOW_SECONDS_CANDIDATES:
    W = int(round(sec * SAMPLING_RATE_HZ))
    for ov in OVERLAP_CANDIDATES:
        S = max(1, int(round(W * (1 - ov))))
        r = runs.copy(); r["n_win"] = n_windows(r["length"].to_numpy(), W, S)
        g = r.groupby(["position", "subject", "label"])["n_win"].sum().rename("n_windows").reset_index()
        rows.append(g.assign(window_s=sec, window_samples=W, overlap=ov, step=S))
win = pd.concat(rows, ignore_index=True)
save_table(win, "window_counts_position_subject_label", index=False)

summ = (win.groupby(["window_s", "overlap", "position"])
          .agg(total_windows=("n_windows", "sum"), min_per_subject_class=("n_windows", "min"),
               n_cells_below_threshold=("n_windows", lambda x: int((x < MIN_WINDOWS_PER_SUBJECT_CLASS).sum())))
          .reset_index())
display(summ)
save_table(summ, "window_feasibility_summary", index=False)

# fraction of rows that cannot form any window (runs shorter than W)
lost = []
for sec in WINDOW_SECONDS_CANDIDATES:
    W = int(round(sec * SAMPLING_RATE_HZ))
    lost.append({"window_s": sec, "rows_in_runs_shorter_than_window_%": round(100 * runs.loc[runs.length < W, "length"].sum() / n_rows, 3)})
lost = pd.DataFrame(lost); display(lost); save_table(lost, "rows_lost_to_short_runs", index=False)

worst = summ.groupby("window_s")["n_cells_below_threshold"].sum()
for sec, n_bad in worst.items():
    flag("PASS" if n_bad == 0 else "WARN", f"Window {sec}s: subject×class×position cells with < {MIN_WINDOWS_PER_SUBJECT_CLASS} windows", str(int(n_bad)))
SUMMARY["window_feasibility"] = summ.to_dict(orient="records")

## 9 · Signal statistics, normalisation pattern, subject shift
The UCI page says the data are *normalized*, but not how. The heuristic below only describes what the numbers look like; confirm against the original RecGym paper before writing the methods section. **Whatever the provided normalisation is, all further scaling in your pipeline must be fitted on training subjects only.**

In [ ]:
q = [0.01, 0.5, 0.99]
stat_rows = []
for p in positions:
    sub = df[df.position == p]
    for k in ALL_CH:
        x = sub[k]
        stat_rows.append({"position": p, "channel": k, "mean": x.mean(), "std": x.std(), "min": x.min(),
                          "p01": x.quantile(q[0]), "median": x.quantile(q[1]), "p99": x.quantile(q[2]), "max": x.max()})
stats = pd.DataFrame(stat_rows); display(stats.round(4)); save_table(stats, "channel_stats_by_position", index=False)

# per subject x position channel mean/std
g = df.groupby(["position", "subject"])[ALL_CH].agg(["mean", "std"])
g.columns = [f"{a}_{b}" for a, b in g.columns]
save_table(g, "channel_stats_by_position_subject")

# --- heuristic: which level looks standardised (mean~0, std~1)?
def near_standardised(m, s, tol=0.05): return bool((np.abs(m) < tol).all() and (np.abs(s - 1) < tol).all())
glob_m, glob_s = df[IMU].mean().to_numpy(), df[IMU].std().to_numpy()
pos_ok = all(near_standardised(df[df.position == p][IMU].mean().to_numpy(), df[df.position == p][IMU].std().to_numpy()) for p in positions)
sub_ok = bool(all(near_standardised(g.loc[(p, s), [f"{k}_mean" for k in IMU]].to_numpy(dtype=float),
                                    g.loc[(p, s), [f"{k}_std" for k in IMU]].to_numpy(dtype=float)) for p in positions for s in subs))
if sub_ok: pattern = "per-subject-and-position standardisation (mean~0, std~1 within every subject/position)"
elif pos_ok: pattern = "per-position standardisation (mean~0, std~1 within each position, not within each subject)"
elif near_standardised(glob_m, glob_s): pattern = "global standardisation across all rows"
else: pattern = "no simple standardisation pattern detected (could be min-max, per-session, or other)"
log("Normalisation pattern (heuristic):", pattern)
SUMMARY["normalisation_pattern_heuristic"] = pattern
flag("INFO", "Normalisation pattern (heuristic, verify in the RecGym paper)", pattern)
if sub_ok:
    flag("WARN", "Data look standardised per subject", "this removes between-subject offset/scale already; state this in the paper's limitations and do not re-fit it using test subjects")

# --- between-subject shift: spread of per-subject channel means in pooled-std units
shift = []
for p in positions:
    gm = g.loc[p]
    pooled = df[df.position == p][IMU].std()
    for k in IMU:
        shift.append({"position": p, "channel": k, "std_of_subject_means_over_pooled_std": float(gm[f"{k}_mean"].std() / pooled[k]),
                      "std_of_subject_stds_over_pooled_std": float(gm[f"{k}_std"].std() / pooled[k])})
shift = pd.DataFrame(shift); display(shift.round(3)); save_table(shift, "between_subject_shift", index=False)

# --- outliers and sample-and-hold (repeated identical values) per channel
out_rows = []
for p in positions:
    sub = df[df.position == p]
    for k in IMU:
        x = sub[k].to_numpy(); m, s = x.mean(), x.std()
        out_rows.append({"position": p, "channel": k,
                         "frac_beyond_5std_%": 100 * float((np.abs(x - m) > 5 * s).mean()),
                         "frac_identical_to_previous_%": 100 * float((x[1:] == x[:-1]).mean())})
outl = pd.DataFrame(out_rows); display(outl.round(3)); save_table(outl, "outliers_and_repeats", index=False)
if (outl["frac_identical_to_previous_%"] > 20).any():
    flag("WARN", "Some channels repeat the previous value in >20% of rows", "possible dropouts / sample-and-hold; inspect outliers_and_repeats.csv")
else:
    flag("PASS", "No channel shows heavy repeated-value behaviour")

# subject-mean heatmap figure
fig, axes = plt.subplots(1, len(positions), figsize=(4.6 * len(positions), 0.5 * len(subs) + 2.5), squeeze=False)
for ax, p in zip(axes[0], positions):
    m = g.loc[p][[f"{k}_mean" for k in IMU]].reindex(subs).to_numpy(dtype=float)
    im = ax.imshow(m, aspect="auto", cmap="coolwarm", vmin=-np.nanmax(np.abs(m)), vmax=np.nanmax(np.abs(m)))
    ax.set_xticks(range(6)); ax.set_xticklabels(IMU); ax.set_yticks(range(len(subs))); ax.set_yticklabels(subs, fontsize=7)
    ax.set_title(f"{p}: per-subject channel mean"); fig.colorbar(im, ax=ax, fraction=0.05)
save_fig(fig, "subject_channel_means")

# distributions per position (sub-sample for speed; descriptive plot only)
rng = np.random.default_rng(SEED)
fig, axes = plt.subplots(len(positions), 6, figsize=(18, 2.6 * len(positions)), squeeze=False)
for i, p in enumerate(positions):
    sub = df[df.position == p]; idx = rng.choice(len(sub), size=min(150_000, len(sub)), replace=False)
    for j, k in enumerate(IMU):
        axes[i, j].hist(sub[k].to_numpy()[idx], bins=80); axes[i, j].set_title(f"{p} · {k}", fontsize=9)
save_fig(fig, "channel_distributions")

## 10 · Example waveforms (one random run per label, per position)
Purely visual sanity check that labels look like motion of the right kind; the random pick is seeded.

In [ ]:
rng = np.random.default_rng(SEED)
for p in positions:
    fig, axes = plt.subplots(len(labels), 1, figsize=(10, 1.45 * len(labels)), sharex=True, squeeze=False)
    for ax, l in zip(axes[:, 0], labels):
        cand = runs[(runs.position == p) & (runs.label == l) & (runs.length >= 10 * SAMPLING_RATE_HZ)]
        if cand.empty: cand = runs[(runs.position == p) & (runs.label == l)]
        if cand.empty:
            ax.set_ylabel(l, rotation=0, ha="right", fontsize=8); continue
        r = cand.iloc[int(rng.integers(len(cand)))]
        seg = df.iloc[int(r.start_row): int(r.start_row) + min(int(r.length), 10 * SAMPLING_RATE_HZ)]
        t = np.arange(len(seg)) / SAMPLING_RATE_HZ
        for k in IMU: ax.plot(t, seg[k].to_numpy(), lw=0.8, label=k)
        ax.set_ylabel(l, rotation=0, ha="right", fontsize=8); ax.tick_params(labelsize=7)
    axes[0, 0].legend(ncol=6, fontsize=7, loc="upper right"); axes[-1, 0].set_xlabel("seconds (nominal)")
    fig.suptitle(f"Example 10-second excerpts — {p}", y=1.0)
    save_fig(fig, f"example_waveforms_{p}")

## 11 · Experimental design files (axis configs, LOSO folds, compute plan)
These are generated from the **actual** subject / position lists so later notebooks read one shared definition.

In [ ]:
acc, gyr = ["Ax", "Ay", "Az"], ["Gx", "Gy", "Gz"]
import itertools
configs = {"6-axis": IMU, "acc-3": acc, "gyro-3": gyr}
for a, b in itertools.combinations(acc, 2): configs[f"acc-{a}{b}"] = [a, b]
for a, b in itertools.combinations(gyr, 2): configs[f"gyro-{a}{b}"] = [a, b]
for k in IMU: configs[f"single-{k}"] = [k]
log("Axis configurations:", len(configs)); [log("  ", k, "->", v) for k, v in configs.items()]
save_json({"configs": configs, "excluded_channel": "C1 (capacitance) — not used in primary experiments", "channels": chan_cols}, "axis_configs.json")

folds = [{"fold": i, "test_subject": s, "train_subjects": [x for x in subs if x != s]} for i, s in enumerate(subs)]
save_json(folds, "loso_folds.json")

N_MODELS = 4  # RF, SVM, XGBoost, 1D CNN
n_fits = len(folds) * len(positions) * len(configs) * N_MODELS
plan = {"n_folds": len(folds), "n_positions": len(positions), "n_configs": len(configs), "n_models": N_MODELS,
        "total_model_fits_primary_grid": n_fits,
        "note": "Nested subject-wise hyper-parameter search (if used) multiplies this further; plan the budget before running."}
save_json(plan, "compute_plan.json"); log(plan)
SUMMARY["compute_plan"] = plan

## 12 · Save helper artefacts (canonical cache + small CLI test sample)

In [ ]:
canon_cols = ["row_idx", "subject", "position", "label", "label_id"] + ALL_CH
canon = df[canon_cols]
if SAVE_CANONICAL_PARQUET:
    try:
        canon.to_parquet(CACHE / "recgym_canonical.parquet", index=False)
        log("Saved canonical parquet to", CACHE / "recgym_canonical.parquet")
    except Exception as e:
        canon.to_pickle(CACHE / "recgym_canonical.pkl"); log("Parquet failed (", e, ") -> pickle saved instead")

# small contiguous sample for local / CLI-agent unit tests (keeps runs contiguous; NOT for any result)
keep = []
for (p, s, l), grp in runs.groupby(["position", "subject", "label"]):
    for _, r in grp.head(CLI_SAMPLE_RUNS_PER_GROUP).iterrows():
        a = int(r.start_row); keep.append((a, a + min(int(r.length), CLI_SAMPLE_ROWS_PER_RUN)))
sample = pd.concat([canon.iloc[a:b] for a, b in sorted(keep)], ignore_index=True)
sample.to_csv(OUT / "cli_test_sample.csv.gz", index=False, compression="gzip")
log("CLI sample rows:", len(sample), "| size MB:", round(os.path.getsize(OUT / "cli_test_sample.csv.gz") / 1e6, 2))

## 13 · Final report, environment record, zip

In [ ]:
import importlib
def ver(m):
    try: return importlib.import_module(m).__version__
    except Exception: return "n/a"
env = {"python": sys.version, "platform": platform.platform(), "seed": SEED,
       "packages": {m: ver(m) for m in ["numpy", "pandas", "matplotlib", "pyarrow"]},
       "data_file": data_file, "data_sha256": DATA_SHA256, "run_timestamp_utc": pd.Timestamp.now("UTC").isoformat()}
save_json(env, "environment.json")
SUMMARY["flags"] = [{"status": s, "check": c, "detail": d} for s, c, d in FLAGS]
save_json(SUMMARY, "audit_summary.json")

n_by = {k: sum(1 for f in FLAGS if f[0] == k) for k in ["PASS", "INFO", "WARN", "FAIL"]}
lines = ["# RecGym Data Audit Report", "",
         f"- Data file SHA-256: `{DATA_SHA256}`", f"- Rows: {n_rows:,}", f"- Subjects: {len(subs)}  |  Positions: {positions}",
         f"- Labels ({len(labels)}): {labels}", f"- Seed: {SEED}  |  Nominal sampling rate used for time conversion: {SAMPLING_RATE_HZ} Hz (from documentation, not verifiable in file)", "",
         f"## Flag summary: {n_by}", ""]
for st in ["FAIL", "WARN", "INFO", "PASS"]:
    for s, c, d in FLAGS:
        if s == st: lines.append(f"- **{s}** — {c}" + (f": {d}" if d else ""))
lines += ["", "## Open decisions for the next notebook (not made here)",
          "- Window length and overlap (justify a priori; see tables/window_feasibility_summary.csv for feasibility only)",
          "- Treatment of the Null class (include in primary analysis, and/or report with and without)",
          "- Whether to use all runs or cap per-class windows",
          "- Scaling: fit on training subjects only inside each fold",
          "- Hyper-parameter search: nested, subject-grouped, inside training subjects only"]
(OUT / "data_audit_report.md").write_text("\n".join(lines))
(OUT / "audit_log.txt").write_text("\n".join(LOG_LINES))
print("\n".join(lines))

# ---- zip
zip_path = BASE / "recgym_00_data_audit_outputs.zip"
if zip_path.exists(): zip_path.unlink()
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for f in OUT.rglob("*"):
        if f.is_file(): z.write(f, f.relative_to(BASE))
    if INCLUDE_CACHE_IN_ZIP:
        for f in CACHE.rglob("*"):
            if f.is_file(): z.write(f, f.relative_to(BASE))
log("ZIP ready:", zip_path, "| MB:", round(os.path.getsize(zip_path) / 1e6, 2))
try:
    from google.colab import files
    files.download(str(zip_path))
except Exception:
    log("Not on Colab (or download blocked). On Kaggle, find the zip in the Output tab of the notebook.")